In [ ]:
%%sql
select * from edw.tbl_fact_policy_quotes limit 100
-- comment by Vijay

StatementMeta(, b24375f6-c340-426e-aeae-01681f026ac9, 2, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 51 fields>

In [ ]:
%%sql
select * from edw.tbl_fact_policy_sales limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_renewals limit 100

In [ ]:
%%sql
select * from dlk.hfq_quotedetails limit 100

In [ ]:
%%sql
select * from dlk.genesys_session_summary 
Where conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and direction = 'outbound' limit 100

Create Event Stream

In [ ]:
Create Table If not Exists  ods.EpisodeEventStream 
(
    EventId     BigInt IDENTITY,
    EventSourceId   Int,
    EventSourceReference    Varchar(128),
    SourceQuoteReference    Varchar(128),
    SourcePolicyReference   Varchar(128),
    SourceCustomerReference Varchar(128),
    SourceSystemId          int,
    PolicyTypeGroup         varchar(50),
    EventDateTime           TIMESTAMP,
    EventDate               Date,
    EventTypeId             VARCHAR(10),
    EventDescription        varchar(1024),
    Grain                   varchar(50)
)

Populate step A1

In [13]:
/* HA1 */ 
Create Or Replace Table stg.HA1_HFQ_Quotes_Raw as 
SELECT  *, 	
        CAST(from_unixtime(_ts) AS TIMESTAMP) as ts_unix,
        ROW_NUMBER() OVER(PARTITION BY QuoteCodeReference, RetrieveCount order by _ts desc) AS RN,
        COUNT(*)     OVER (PARTITION BY QuoteCodeReference, RetrieveCount) AS MaxRN
FROM    dlk.HFQ_QuoteDetails_Snapshot_v2
Where   CAST(from_unixtime(_ts) AS TIMESTAMP) between '2026-07-01' and '2026-07-31'  
;

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
/* HA1 */ 
Create Or Replace Table stg.HA1_HFQ_Quotes as 
SELECT  *
FROM    (SELECT  *, 	
                CAST(from_unixtime(_ts) AS TIMESTAMP) as ts_unix,
                ROW_NUMBER() OVER(PARTITION BY QuoteCodeReference, RetrieveCount order by _ts desc) AS RN,
                COUNT(*)     OVER (PARTITION BY QuoteCodeReference, RetrieveCount) AS MaxRN
        FROM    dlk.HFQ_QuoteDetails_Snapshot_v2) a
where   RN =1
and     ts_unix between '2026-07-01' and '2026-07-31'  

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
select count(*) from stg.HA1_HFQ_Quotes

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [5]:
select * from dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Home % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType in ('EMAIL' , 'SMS')
limit 10

StatementMeta(, 2d922262-ae48-408d-b50e-68a51f32d4d7, 6, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [6]:
select * from stg.HA1_HFQ_Quotes limit 10

StatementMeta(, 2d922262-ae48-408d-b50e-68a51f32d4d7, 7, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 111 fields>

In [9]:
select * from stg.HA1_HFQ_Quotes a join (select * from dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Home % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType = 'EMAIL' ) b
on a.proposer_email = b.email limit 10

StatementMeta(, 2d922262-ae48-408d-b50e-68a51f32d4d7, 10, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 128 fields>

In [10]:
/*HA1.F1*/
select count(*) from stg.HA1_HFQ_Quotes a left join (select * from dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Home % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType = 'EMAIL' ) b
on a.proposer_email = b.email
where b.QuoteQueryGuid is null

StatementMeta(, 2d922262-ae48-408d-b50e-68a51f32d4d7, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [8]:
/* HA1.E1
 */
select	count(*)
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Home % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType = 'EMAIL'  --> this brings down to 23058
-- and QuoteQueryGuid > ''  --> this brings down to 12148 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## Populate step A0

In [9]:
Create or Replace Table stg.A0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:

Create or Replace Table stg.A0_genesys_derived_data_filtered as 
select	a.ConversationId, CustomerPhoneNumber,a.sessionIndex,A.conversationStartTime
from	stg.A0_genesys_derived_data	a,
		(select	ConversationId, max(sessionIndex) sessionIndex
		from	stg.A0_genesys_derived_data
		Where	queueName is not null 
		and		originatingDirection = 'inbound'
		Group by ConversationId
		)	b 
where	a.ConversationId = b.ConversationId
and		a.sessionIndex = b.sessionIndex
and		queueName = 'INBOUND_SALES_HOME'
 

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
/* HA1.E2
 */ 
select	count(distinct CustomerPhoneNumber) 
from	stg.A0_genesys_derived_data	a,
		(select	ConversationId, max(sessionIndex) sessionIndex
		from	stg.A0_genesys_derived_data
		Where	queueName is not null 
		/*and		originatingDirection = 'inbound'*/
		Group by ConversationId
		)	b 
where	a.ConversationId = b.ConversationId
and		a.sessionIndex = b.sessionIndex
and		queueName = 'OUTBOUND_SALES_HOME'

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [15]:
Create or Replace Table stg.A0_hfq_derived_data as 
select distinct 
    replace(proposer_mobilePhoneNumber, '+353', '0') as CustomerPhoneNumber,
    QuoteCodeReference 
from
    dlk.hfq_quotedetails_snapshot_v2
Where  CAST(from_unixtime(_ts) AS TIMESTAMP)  between '2026-07-01' and '2026-07-31'    


StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [20]:
Create or Replace table stg.A0_HFQ_genesys_link as 
select  ConversationId,SessionIndex 
from    stg.A0_genesys_derived_data_filtered a left join stg.A0_hfq_derived_data b
        on a.CustomerPhoneNumber = b.CustomerPhoneNumber
where   b.CustomerPhoneNumber is null
Group by ConversationId,SessionIndex
;

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [21]:
/* HA0
*/
SELECT count(distinct  ConversationId)
FROM
    stg.A0_HFQ_genesys_link

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 22, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [23]:
Create or Replace Table stg.HA0F1_genesys_inbound_call_duration_summary as 
select	a.ConversationId, min(conversationStartTime) as conversationStartTime, 
            sum(agentAnswered) as agentAnswered, sum(alertNoAnswer) alertNoAnswer, sum(abandoned) abandoned, 
            sum(totalAcdWaitDuration) CallWaitTime, sum(totalAgentAlertDuration) CallRingTime, sum(totalAgentHoldDuration) CallHoldTime, sum(totalAgentTalkDuration) CallSpokenTime
from	stg.A0_genesys_derived_data	a
where	a.queueName = 'INBOUND_SALES_HOME'  
Group by ConversationId

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
/* HA0.F1
*/
select count(*) from stg.HA0F1_genesys_inbound_call_duration_summary where abandoned = 1 and CallWaitTime > 0 limit 10

StatementMeta(, e9399029-f72d-45fb-959a-4c90cf888ec3, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [16]:
/* HA2
 */
 --old way, gives a much smaller number
--select count(distinct QuoteCodeReference) from stg.HA1_HFQ_Quotes Where MaxRN <> RN 

select 
count(distinct QuoteCodeReference) 
--*
from dlk.hfq_response_quotes
where Quotes_DateCreated between '2026-07-01' and '2026-07-31'

StatementMeta(, 036f952a-f77d-4fb0-bd15-dd13950bac30, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
Create or Replace Table stg.hfq_price_requested as 
select  QuoteCodeReference 
from    dlk.HFQ_Response_Quotes 
Group by QuoteCodeReference

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
Create or Replace Table stg.hfq_prices as 
select  QuoteCodeReference ,  min(Quotes_TotalPayable) as QuoteBestPrice
from    dlk.HFQ_Response_Quotes 
where   Quotes_Premium <> 0 
and     Quotes_Outcome = 'PremiumReturned'
Group by QuoteCodeReference

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
/* HA2.F1
*/
select count(distinct a.QuoteCodeReference) from stg.HA1_HFQ_Quotes a left join stg.hfq_price_requested b 
on a.QuoteCodeReference = b.QuoteCodeReference
Where b.QuoteCodeReference is null 

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [14]:
select * from stg.HA1_HFQ_Quotes_Raw limit 10 

StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 111 fields>

In [ ]:
Create or Replace table stg.HA3_HomeAcquisitionQuoteInitiated as 
select  QuoteQueryGuid, 
        max(RetrieveSessionToken) RetrieveSessionToken,
        min(case when StepId = 1 then CreatedOn else null end) Step1DateTime,
        max(case when StepId in (2,3,4,5) then StepId else null end) Step2To5,
        min(case when StepId in (2,3,4) then CreatedOn else null end) FirstStep2to4DateTime,
        min(case when StepId = 5 then CreatedDateCreatedOnTime else null end) FirstStep5DateTime,
        max(case when StepId = 5 then CreatedOn else null end) LastStep5DateTime, 
        max(BrokerId) BrokerId,
        max(case when ImpervalsBot = 'true' then 1 else 0 end) ImpervalsBot,
        max(case when ImpervalsBot = 'true' then 1 else 0 end) ImpervalsBot
from    dlk.hfq_quotedetails_snapshot_v2
Where   CreatedOn between '2026-07-01' and '2026-07-31'
Group by QuoteQueryGuid
;

In [12]:
--HA3
select 
count(distinct QuoteCodeReference) 
--*
from dlk.hfq_response_quotes
where Quotes_Premium > 0
and Quotes_Outcome = 'PremiumReturned'
and Quotes_DateCreated between '2026-07-01' and '2026-07-31'
--limit 10

StatementMeta(, 036f952a-f77d-4fb0-bd15-dd13950bac30, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [14]:
select count(distinct QuoteCodeReference) 
from
dlk.hfq_response_quotes
where 
--Quotes_Premium > 0
--and Quotes_Outcome = 'Refer'
Quotes_DateCreated between '2026-07-01' and '2026-07-31'
limit 10

StatementMeta(, 036f952a-f77d-4fb0-bd15-dd13950bac30, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
select * from dlk.hfq_response_quotes
where 
--Quotes_Premium > 0
--and Quotes_Outcome = 'Refer'
Quotes_DateCreated between '2026-07-01' and '2026-07-31' limit 10

StatementMeta(, 8f798dea-f195-4411-bdff-648ba4041eeb, 3, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 31 fields>

In [6]:
/* HFQ payment ODS event joins. Read-only; mirrors DWH.usp_Load_fact_HFQ_Payment. */
Create or Replace Table stg.HFQ_Payments as 
WITH Request_Request AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(globalPayment_reference)) AS PaymentReference, trim(upper(reference)) AS QuoteReference,
               SchemeCode, TemporaryQuoteId, CacheId, amount, depositAmount,
               isFullPayment, monthlyAmount, payer_email, policyStartDate, BrokerID,
               ROW_NUMBER() OVER
                 (PARTITION BY globalPayment_reference ORDER BY globalPayment_reference DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Request
        WHERE SourceFilePath NOT LIKE '%Global%'
          AND TemporaryQuoteId IS NOT NULL AND CacheId IS NOT NULL
          AND globalPayment_reference IS NOT NULL AND reference IS NOT NULL
    ) AS source_data
    WHERE rn = 1
),
Request_Response AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(REPLACE(SUBSTRING(SourceFilePath, 26, LEN(SourceFilePath)), '.json', ''))) AS PaymentReference,
               action_time_created, action_result_code, action_type,
               ROW_NUMBER() OVER
                 (PARTITION BY REPLACE(SUBSTRING(SourceFilePath, 26, LEN(SourceFilePath)), '.json', '')
                  ORDER BY action_time_created DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Response
    ) AS source_data
    WHERE rn = 1
),
Response_Response AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(reference)) AS PaymentReference, action_result_code, action_time_created,
               type, status, amount, payment_method_result, payment_method_message,
               ROW_NUMBER() OVER (PARTITION BY reference ORDER BY action_time_created DESC) AS rn
        FROM dlk.HFQ_PaymentResponse_Response
    ) AS source_data
    WHERE rn = 1
),
Request_Failed AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(REPLACE(SUBSTRING(SourceFilePath, 24, LEN(SourceFilePath)), '.json', ''))) AS PaymentReference,
               StatusCode, ReasonPhrase, IsSuccessStatusCode,
               REPLACE(REPLACE(Headers_Value, '["', ''), '"]', '') AS FailureTimestamp,
               ROW_NUMBER() OVER
                 (PARTITION BY REPLACE(SUBSTRING(SourceFilePath, 24, LEN(SourceFilePath)), '.json', '')
                  ORDER BY Headers_Value DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Failed
    ) AS source_data
    WHERE rn = 1
)
SELECT
    req.PaymentReference AS RequestPaymentReference,
    req.QuoteReference, req.BrokerID, req.SchemeCode, req.TemporaryQuoteId, req.CacheId,
    req.amount AS RequestAmount, req.depositAmount, req.isFullPayment, req.monthlyAmount,
    req.payer_email, req.policyStartDate,
    reqRes.PaymentReference AS RequestResponsePaymentReference,
    reqRes.action_time_created AS RequestResponseTimestamp,
    reqRes.action_result_code AS RequestResponseActionResult,
    reqRes.action_type AS RequestResponseActionType,
    resRes.PaymentReference AS PaymentResponsePaymentReference,
    resRes.action_result_code AS PaymentResponseActionResultCode,
    resRes.action_time_created AS PaymentResponseTimestamp,
    resRes.type AS PaymentResponseType, resRes.status AS PaymentResponseStatus,
    resRes.amount AS PaymentResponseAmount,
    resRes.payment_method_result AS PaymentMethodResult,
    resRes.payment_method_message AS PaymentMethodMessage,
    fail.PaymentReference AS FailedRequestPaymentReference,
    fail.StatusCode AS FailedRequestStatusCode, fail.ReasonPhrase AS FailedRequestReason,
    fail.IsSuccessStatusCode AS FailedRequestIsSuccessStatusCode, fail.FailureTimestamp,

    /* Fact flags and dates, named to match DWH.tbl_fact_HFQ_Payment. */
    CASE WHEN fail.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentRequestFailed,
    fail.FailureTimestamp AS PaymentRequestFailedTimestamp,
    CASE WHEN reqRes.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentRequestResponse,
    reqRes.action_time_created AS PaymentRequestResponseTimestamp,
    CASE WHEN resRes.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentResponseResponse,
    resRes.action_result_code AS PaymentResponseResponseActionResultCode,
    resRes.action_time_created AS PaymentResponseResponseTimestamp,
    CASE WHEN resRes.action_result_code IS NOT NULL THEN 1 END AS RecordVersionIsLatestFlag,
    CASE
        WHEN resRes.action_time_created IS NOT NULL THEN CAST(resRes.action_time_created AS date)
        WHEN reqRes.action_time_created IS NOT NULL THEN CAST(reqRes.action_time_created AS date)
        WHEN fail.FailureTimestamp IS NOT NULL THEN 
                            cast(concat(
                                SUBSTRING(fail.FailureTimestamp, 13, 4) , '-' ,
                                CASE SUBSTRING(fail.FailureTimestamp, 9, 3)
                                    WHEN 'Jan' THEN '01' WHEN 'Feb' THEN '02' WHEN 'Mar' THEN '03'
                                    WHEN 'Apr' THEN '04' WHEN 'May' THEN '05' WHEN 'Jun' THEN '06'
                                    WHEN 'Jul' THEN '07' WHEN 'Aug' THEN '08' WHEN 'Sep' THEN '09'
                                    WHEN 'Oct' THEN '10' WHEN 'Nov' THEN '11' WHEN 'Dec' THEN '12'
                                        END , '-' , SUBSTRING(fail.FailureTimestamp, 6, 2), ' ',
                                        Substring(fail.FailureTimestamp,18,8)) 
                                        as timestamp)
    END AS WorkDate
FROM Request_Request AS req
FULL OUTER JOIN Request_Response AS reqRes
    ON req.PaymentReference = reqRes.PaymentReference
FULL OUTER JOIN Response_Response AS resRes
    ON req.PaymentReference = resRes.PaymentReference
FULL OUTER JOIN Request_Failed AS fail
    ON req.PaymentReference = fail.PaymentReference;


StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
/* HA4a
*/
select count(distinct a.QuoteCodeReference) from stg.HA1_HFQ_Quotes a join stg.hfq_payments b 
on a.QuoteCodeReference = b.QuoteReference
Where b.PaymentResponseResponseActionResultCode = 'SUCCESS' and 
PaymentResponseResponseTimestamp   between '2026-07-01' and '2026-07-31' 

StatementMeta(, 501395ed-8c6c-4fb7-9f27-f0669b21c357, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [9]:
/*
HA4b
Home	Acquisition	HA4.F1
*/
SELECT 
a.channel,
COUNT(distinct a.PolicyCode) 
--*
FROM stg.GlobalPoliciesSold      a
Where   a.ReportingSaleType = 'New Business' 
and     a.PolicyTypeGroup = 'Home'
and     a.QuoteQueryGuid is null 
group by a.channel
--limit 10
;

StatementMeta(, 6d8ad5d1-6aff-43a9-af00-6c6089142a22, 10, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [1]:
/*HA5*/
select 
--* 
count(distinct QuoteReference)
--count(*)
from stg.HFQ_payments where PaymentResponseResponseActionResultCode = 'SUCCESS' and 
PaymentResponseResponseTimestamp   between '2026-07-01' and '2026-07-31' 
--limit 10

StatementMeta(, 501395ed-8c6c-4fb7-9f27-f0669b21c357, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [11]:
/*HA5.F1*/
select 
count(distinct QuoteReference)
from stg.HFQ_payments 
where 
PaymentResponseResponseActionResultCode = 'DECLINED' 
and PaymentResponseResponseTimestamp   between '2026-07-01' and '2026-07-31' 
--limit 10

StatementMeta(, f4b95894-f0c7-495d-9f07-76f06bfbe123, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [18]:
/*HA5.B1*/
select 
--*
--PaymentResponseType,
count(*)
from stg.HFQ_payments 
where 
--PaymentResponseResponseActionResultCode = 'DECLINED' 
PaymentResponseResponseTimestamp   between '2026-07-01' and '2026-07-31' 
and isFullPayment = 'false'
--limit 100

StatementMeta(, f4b95894-f0c7-495d-9f07-76f06bfbe123, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
/*HA5.1*/
SELECT
count(distinct a.PolicyCode)
       --count(distinct a.PolicyCode)
from    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3
and     d.PolicyTypeGroup = 'Home'
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
and     ReportingSaleType = 'New Business'
and     EventDescription in
            (
                'New Business - Emailed Document - Terms Of Business',
                'New Business - Document Transmitted - Terms Of Business',
                'New Business - Emailed Document - Chill Terms of Business'
            );

StatementMeta(, b03c2e2f-a082-4d1e-b8a6-e4e76ef6e406, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
select * from dlk.MyChill_NewUploadDocumentEvents limit 10

StatementMeta(, b03c2e2f-a082-4d1e-b8a6-e4e76ef6e406, 3, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 13 fields>

In [5]:
--
SELECT
EventDescription,count(*)
       --count(distinct a.PolicyCode)
from    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3
and     d.PolicyTypeGroup = 'Home'
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'
and     ReportingSaleType = 'New Business'
group BY
EventDescription
order by 2 desc
;



StatementMeta(, b03c2e2f-a082-4d1e-b8a6-e4e76ef6e406, 6, Finished, Available, Finished, False)

<Spark SQL result set with 320 rows and 2 fields>

In [2]:
/*HA6*/

select 
		count(distinct PolicyCode)  
		from edw.exp_mychill_chase_daily_snapshot_home a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and     PolicyType = 'New Business' 
and     Campaign = 'DAY 1'
and 
case 
	when Gap_In_Cov_Ltr_Status = 'O' then 1 
	when Val_For_Spec_Item_Status = 'O' then 1 
	when PPS_Num_Status = 'O' then 1 
	When Identification_Status = 'O' then 1 
	when Finance_Form_Status = 'O' then 1 
	when Digital_Journey_Status = 'O' then 1 
	else 0
End = 1

StatementMeta(, bc456483-3555-46ef-82af-562f10336593, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
/*HA6.F1*/

--requires clarity

StatementMeta(, bc456483-3555-46ef-82af-562f10336593, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
/* HA7 */
select count(distinct PolicyCode)
FROM    stg.globalpoliciessold                   a,
        ods.eventstream                          d
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3
and     d.PolicyTypeGroup = 'Home'
and     EventDateTime > '2026-07-01 00:00:00.000'
and     ReportingSaleType = 'New Business'
and     EventDescription in 
(
'New Business - Emailed Document - Terms Of Business',
'New Business - Document Transmitted - Terms Of Business',
'New Business - Emailed Document - Chill Terms of Business',
'New Business - Document Transmitted - Chill Terms of Business',
'New Business - Emailed Document - 05E Terms Of Business')

StatementMeta(, 9caaadb8-099d-4962-9b29-1269ac6d0023, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
/*HA7.B1.2*/
select count(distinct PolicyCode)
FROM    stg.globalpoliciessold                   a,
        ods.eventstream                          d
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3
and     d.PolicyTypeGroup = 'Home'
and     EventDateTime > '2026-07-01 00:00:00.000'
and     ReportingSaleType = 'New Business'
and     (
            EventDescription like 'New Business - Emailed Document - %'     
        or  EventDescription like 'New Business - Document Transmitted - %' 
            )
and (
   EventDescription like '%04 - Suitability Statement'
Or EventDescription like '%05E Terms Of Business'
Or EventDescription like '%205 SOF- Proposal_Placeholder'
Or EventDescription like '%300 NB Home Email'
Or EventDescription like '%Chill Terms of Business'
Or EventDescription like '%Cover Letter'
Or EventDescription like '%CV Checklist Link'
Or EventDescription like '%CV Cover Letter'
Or EventDescription like '%CV Email Body'
Or EventDescription like '%CV Suitability Statment.'
Or EventDescription like '%Direct Debit Mandate'
Or EventDescription like '%Document Checklist Email'
Or EventDescription like '%Document Checklist SMS'
Or EventDescription like '%Gap In Cover Letter'
Or EventDescription like '%Home Cover Letter'
Or EventDescription like '%Home LOI_Placeholder'
Or EventDescription like '%Home Schedule_Placeholder'
Or EventDescription like '%Home SOF / Proposal_Placeholder'
Or EventDescription like '%Home Suitability Statement'
Or EventDescription like '%Insurance Product Information Document'
Or EventDescription like '%InsuranceProductInformationDocument'
Or EventDescription like '%Insurer Doc 1_Placeholder'
Or EventDescription like '%Insurer Doc 2_Placeholder'
Or EventDescription like '%Insurer Doc 3_Placeholder'
Or EventDescription like '%Interested Party'
Or EventDescription like '%Interested Party - Allied Irish Bank plc(AIB Mortgage Bank)'
Or EventDescription like '%Interested Party - Bank of Ireland (2 College Green)(Bank of Ireland)'
Or EventDescription like '%Interested Party - Bank of Ireland Mortgage Bank U.C.(College Green)'
Or EventDescription like '%Interested Party - Bank of Ireland Mortgages(Dublin - 2 College Green)'
Or EventDescription like '%Interested Party - Bank of Ireland(College Green)'
Or EventDescription like '%Interested Party - Bankinter SA(Maynooth)'
Or EventDescription like '%Interested Party - EBS D.A.C(Artane)'
Or EventDescription like '%Interested Party - Haven Mortgages Limited(Head Office)'
Or EventDescription like '%Interested Party - Permanent TSB(Dublin - Head Office)'
Or EventDescription like '%Interested Party - Permanent TSB(Head Office)'
Or EventDescription like '%InterestedParty'
Or EventDescription like '%Key Facts'
Or EventDescription like '%Premium Breakdown'
Or EventDescription like '%PremiumBreakdown'
Or EventDescription like '%Quote Breakdown'
Or EventDescription like '%Receipt'
Or EventDescription like '%Receipt_Template'
Or EventDescription like '%Schedule'
Or EventDescription like '%Statement Of Fact'
Or EventDescription like '%StatementOfFact'
Or EventDescription like '%Terms Of Business'
)

In [11]:
/*select * from stg.HA1_HFQ_Quotes 
where RN <> MaxRN limit 100 ;
*/
select * from dlk.hfq_quotedetails_snapshot_v2 
Where RetrieveCount > 0 


StatementMeta(, 87866fa8-457b-4404-9eff-1f38a121070e, 12, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 111 fields>

In [103]:
/* HFQ / MFQ to Policy links to be derived properly - we are picking up from DWH for now */
Create or Replace Table stg.GlobalPoliciesSold as 
select  PolicyCode, s.Channel, s.ReportingSaleType, s.PolicyTypeGroup, QuoteQueryGuid, HEQReference, ReportingSaleDate 
from    edw.tbl_fact_policy_sales    s 
WHERE   s.EffectiveDate = '2026-07-31'
  AND   s.ReportingSaleCategory = 'Cat A1: Active Sale'
  AND   s.OrgID = 1
  AND   s.SaleCount = 1
  ;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 104, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [104]:
/* Renewals and sales to be derived properly - we are picking up from DWH for now */
/* Renewals are always derived for the month of renewals -- and their sales over time across months */ 
Create or Replace Table stg.R0_MotorPoliciesEligibleForRenewals as 
select  PolicyCode, LyPolicyRenewDateAdj as RenewalDate, TyPolicyCode , TyPolicyRenewDateAdj
from    edw.tbl_fact_policy_renewals    a 
Where   a. LyPolicyTypeGroup = 'Home' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
and     PolicyOfferNum = 1 
Group by   PolicyCode, LyPolicyRenewDateAdj, TyPolicyCode  , TyPolicyRenewDateAdj

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 105, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
SELECT
        ReportingSaleType
        ,count(distinct a.PolicyCode)
from    stg.GlobalPoliciesSold a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
--and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Home'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
group by ReportingSaleType
order by count(*) desc

StatementMeta(, 8f798dea-f195-4411-bdff-648ba4041eeb, 7, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [45]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A4a',
    'Motor Acquisition - Customer buys ONLINE',
    'Quote'
-- A3 query-- we need to copy the SL10 quote query guids as the scraperflag is quite complex than defined in the framework / chill's explanations in the past /* teams message sent to Nikita for check */
from    stg.GlobalPoliciesSold      a,
        (select QuoteQueryGuid, max(PaymentDate) PaymentDate
        from    stg.MFQ_Quote_Payments 
        where   PaymentStatus like '%|Success, code: 00%' 
        and     PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        and     PaymentType in (0,5)
        Group by QuoteQueryGuid) b 
Where   a.QuoteQueryGuid = b.QuoteQueryGuid
;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 46, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [97]:
/* this is not used - MFQ payment to Web Assist resolution needs more analysis -- one for Vijay for Monday */
select  count(distinct PolicyCode), s.Channel 
from    stg.GlobalPoliciesSold      s
left join
        (select QuoteQueryGuid, max(PaymentDate) PaymentDate
        from    stg.MFQ_Quote_Payments 
        where   PaymentStatus like '%|Success, code: 00%' /* remove date filter to eliminate pre or post month purchases */ 
        and     PaymentType in (0,5)
        Group by QuoteQueryGuid) b 
        on   s.QuoteQueryGuid = b.QuoteQueryGuid
Where   b.QuoteQueryGuid is null 
and     s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is not null 
Group by Channel
limit 100 

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 98, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
select  Channel, count(distinct PolicyCode), count(distinct QuoteQueryGuid)
from    stg.GlobalPoliciesSold      s
Where   s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is not null 
Group by Channel
limit 100 

In [ ]:
select  Channel, count(distinct PolicyCode), count(distinct QuoteQueryGuid)
from    stg.GlobalPoliciesSold      s
Where   s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is null 
Group by Channel
limit 100 

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.PolicyCode,
    9,
    'Motor',
    date_trunc('MINUTE', ReportingSaleDate),
    cast(ReportingSaleDate as date),
    'A4b',
    'Motor Acquisition - Agent quotes and converts on the CALL',
    'Policy'
from    stg.GlobalPoliciesSold      a
Where   a.ReportingSaleType = 'New Business' 
and     a.PolicyTypeGroup = 'Motor'
and     a.QuoteQueryGuid is null 
;


In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourcePolicyReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    a.PolicyCode,
    9,
    'Motor',
    date_trunc('MINUTE', ReportingSaleDate),
    cast(ReportingSaleDate as date),
    'A4.F1',
    'Motor Acquisition - Web assist: starts online, then calls to complete',
    'Policy' /* can be quote or policy! */
from    stg.GlobalPoliciesSold      a
Where   a.ReportingSaleType = 'New Business' 
and     a.PolicyTypeGroup = 'Motor'
and     a.QuoteQueryGuid is not null 
and     a.Channel = 'b) Web Assist'
;

In [107]:
select  count(distinct QuoteQueryGuid)
from    stg.MFQ_Quote_Payments 
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 108, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5a',
    'Motor Acquisition - Payment attempted ONLINE',
    'Quote'
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

In [108]:
select  count(distinct QuoteQueryGuid)
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType not in (0,5)
;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 109, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5.F1',
    'Motor Acquisition - Payment failed',
    'Quote'
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType not in (0,5)
;

In [19]:
/* A5.B1 Loan requested in MFQ payments */ 
select  count(distinct QuoteQueryGuid) 
from    stg.MFQ_Quote_Payments
Where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType in (0,5)
and     FullPaymentFlag is FALSE 

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5.B1',
    'Motor Acquisition - Loan setup with the finance provider',
    'Quote'
from    stg.MFQ_Quote_Payments
Where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType in (0,5)
and     FullPaymentFlag is FALSE 
;

In [22]:
select count(*) from dlk.SQL07CDA_Messages

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [36]:
create or Replace table stg.CDAPurchasePolicyList as 
select  trim(PolicyNumber) PolicyCode , max(UTCDateUpdated) SentDate
from    dlk.SQL07CDA_Messages
where	RenderedMessage like 'Hi, thanks for choosing us, we really appreciate it.%be sending out your welcome pack shortly. Thanks, Chill Insurance.' 
and		UTCDateAdded	between  '2026-07-01 00:00:00' and '2026-08-01 00:00:00' 
and     ProviderResponse = 'Delivered'
Group by trim(PolicyNumber)

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 37, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [37]:
select  PolicyTypeGroup, count(*) 
from    stg.GlobalPoliciesSold a, stg.CDAPurchasePolicyList b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
Group by PolicyTypeGroup

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 38, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', SentDate),
        cast(SentDate as date),
        'A5.1',
        'Motor Acquisition - Purchase confirmation sent',
        'Policy'
from    stg.GlobalPoliciesSold a, stg.CDAPurchasePolicyList b  /* the message is already filtered for new business */
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
;

In [34]:
/* Service Retrieve A1.E1 email 

    Ewan - please add this to event stream 
         
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? 
*/
/*
select	*
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Motor - TYR 2.2 - Service Retrieve - Quote Expiry Journey - 0DTI' and interaction_type = 'sent' 
-- without the below conditions we get 76836 entries but it includes workfl/ sms / email so I think only 2/3 rd contacts sent out
and MessageType = 'EMAIL'  --> this brings down to 25269
and QuoteQueryGuid > ''  --> this brings down to 13339 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

the full set of campaign name for July 
Motor - TYR 2.2 - Service Retrieve - Quote Expiry Journey - 0DTI
Motor - TYR 3.0 - Step 3 Complete
Motor - TYR 3.0 - Step 4 Complete
Motor - TYR 3.0 - Step 5 Complete
Motor TYR 2.2 - Buy Now - Service 
Motor TYR 2.2 - Buy Now - Service  AB TEST
Motor TYR 2.2 - Complete Quote - Marketing 
Motor TYR 2.2 - Complete Quote - Service 
Motor TYR 2.2 Buy Now - Marketing 

*/
select	count(*)
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
-- and MessageType = 'EMAIL'  --> this brings down to 23058
-- and QuoteQueryGuid > ''  --> this brings down to 12148 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 35, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
select 
--QuoteQueryGuid,count(*) 
--count(*)
*
from dlk.EXT_XtremePushResults 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and     interaction_type = 'sent' 
and     MessageType = 'EMAIL'
and     QuoteQueryGuid > ''
--and     QuoteQueryGuid = 'a504caec-6a11-492b-a6e6-de2f874e9467'
--group by QuoteQueryGuid
--having count(*) > 1 
limit 10

StatementMeta(, 513eeba3-c789-4544-813e-13631d9ec9b5, 8, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.QuoteQueryGuid,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A1.E1',
        'Motor Acquisition - Service retrieve',
        'Policy'
from    dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and     interaction_type = 'sent' 
and     MessageType = 'EMAIL'
and     QuoteQueryGuid > ''
;

In [13]:
/* purchase confirmation email from Amber A5.1 - email confirmation -- this pulls 2890 ecs and aligns with Nikita's number


    Ewan - please add this to event stream 
        
        but policy code / client code is sparsely populated 
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? */

select	
*
--*
from	dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Online Motor Bind Confirmation' and interaction_type = 'sent' 
--and     QuoteQueryGuid > ''
limit 10

StatementMeta(, 513eeba3-c789-4544-813e-13631d9ec9b5, 14, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [ ]:
--TBC

insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.QuoteQueryGuid,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A5.1',
        'Motor Acquisition - Service retrieve',
        'Policy'
from    dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Online Motor Bind Confirmation' and interaction_type = 'sent' 
;

In [1]:
/* A6.B1.1 is not recorded anywhere - the document checklist sms / email is all we can see -- A6 */
/* A6.B1.2 is here */
select  --count(distinct b.PolicyCode)
*
from    stg.GlobalPoliciesSold a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
limit 100
;

StatementMeta(, b03c2e2f-a082-4d1e-b8a6-e4e76ef6e406, 2, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 20 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A6.B1.2',
        'Motor Acquisition - Customer submits documents',
        'Policy'
from    stg.GlobalPoliciesSold a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
;

In [12]:
/* A6.B1.3 */
select  isAccepted, count(distinct b.PolicyCode)
from    stg.GlobalPoliciesSold a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted
limit 100
;

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 13, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A6.B1.3',
        'Motor Acquisition - Customer submits documents',
        'Policy'
from    stg.GlobalPoliciesSold a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
and     isAccepted = 'true'
;

In [2]:
select  *from dlk.cda_loanse

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 23 fields>

In [3]:
select count(*) from dlk.rbsdata2_diary

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
/* A6.B1.1 A6 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email',
                'New Business - Saved Document - Terms Of Business',    
                'New Business - Sent To Document Processing - Terms Of Business'
            )


StatementMeta(, d5042f13-38fc-46ce-b16d-50aeacbe2afe, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email',
                'New Business - Saved Document - Terms Of Business',
                'New Business - Sent To Document Processing - Terms Of Business'
            )
Group by a.PolicyCode

In [14]:
/* A6.B1.5 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%' 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6.B1.5',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%' 

In [19]:
/* A6.B1.O1 -- no doc escalation and then cancellation */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.PolicyCode in 
        (SELECT  distinct a.PolicyCode
        FROM    stg.GlobalPoliciesSold                   a,
                ods.EventStream                          d 
        Where   a.PolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Motor'
        and     EventDateTime > '2026-07-01 00:00:00.000' 
        and     ReportingSaleType = 'New Business' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6.B1.5',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.PolicyCode in 
        (SELECT  distinct a.PolicyCode
        FROM    stg.GlobalPoliciesSold                   a,
                ods.EventStream                          d 
        Where   a.PolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Motor'
        and     EventDateTime > '2026-07-01 00:00:00.000' 
        and     ReportingSaleType = 'New Business' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 

In [24]:
/* A7 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            ( 
                'New Business - Document Transmission Confirmed - Certificate',
                'New Business - Document Posted Confirmation - Certificate',
                'New Business - Document Transmitted - Certificate'
            )   


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            ( 
                'New Business - Document Transmission Confirmed - Certificate',
                'New Business - Document Posted Confirmation - Certificate',
                'New Business - Document Transmitted - Certificate'
            )   

In [30]:
/* A7 , A7.B1.1 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription IN 
            ( 
                'New Business - Document Transmitted - Certificate' /* as confirmed by 2003 / 2004 / 2014 only this exists --Document Transmitted */
            )   


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.1',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription IN 
            ( 
                'New Business - Document Transmitted - Certificate' /* as confirmed by 2003 / 2004 / 2014 only this exists --Document Transmitted */
            )   

In [29]:
/* A7 , A7.B1.2 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 30, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.2',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.3',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )

In [31]:
/* A7 , A7.B1.2 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'Duplicate Certificate - Document Transmitted - Certificate'
            )

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 32, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.5',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'Duplicate Certificate - Document Transmitted - Certificate'
            )

In [ ]:
-- get chase emails from XP table
select campaign_name from ods.EXT_XtremePushResults where campaign_name like '%Chase%' 
and [timestamp] between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by campaign_name
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Home - Post Renewal &amp; Purchase - xSell 
Home - Renewals Chase - 10 DTI
Home - Renewals Chase - 10 DTI - Ex-Chrome
Home - Renewals Chase - 20DTI
Home - Renewals Chase - 20DTI - Ex-Chrome
Home - Renewals Chase - 2DTI
Home - Renewals Chase - 2DTI - Exh-Chrome
Motor - Post Renewal & Purchase - xSell
Motor - Renewals Chase - 10DTI
Motor - Renewals Chase - 20DTI
Motor - Renewals Chase - 2DTI
Renewal Chase 16 Fri
Renewal Chase 16 Mon-Thur
Renewal Chase 21 Fri
Renewal Chase 21 Mon-Thur
SurveyPurchased Auto PIF
SurveyPurchased July23
Van - Renewals Chase - 10DTI
Van - Renewals Chase - 20DTI
Van - Renewals Chase - 2DTI
*/

In [ ]:
select * from stg.XPTempExtracts where 'Commercial'

        create or replace table tmp.lifelivepolicies as
        select  cast(a.PK_ContactID as varchar(255)) ClientCode,
                b.PK_ContactPolicyID
        from    dlk.life_tblcontacts        a,
                dlk.life_tblcontactpolicies b
        Where   a.PK_ContactID = b.FK_ContactID
        and     FK_PolicyStatusID = 1
        and     PolicyInceptionDate < '2026-08-29 00:00:00.000'
        AND     PolicyInceptionDate <> '1900-01-01 00:00:00.000'
        AND     PolicyMaturityDate >= '2026-08-28'
        AND     PolicyNumber IS NOT NULL
        AND     PolicyNumber <> ''
        AND     PolicyNumber <> 'GDPR'
        Order by 1

In [1]:
select	* 
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Limit 100 

StatementMeta(, 7f34cdd0-8f5e-454d-868f-ab2f23debe7f, 2, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '100'.(line 1, pos 11)

== SQL ==
select	top 100 * 
-----------^^^
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

Hi Lorraine, we have emailed important correspondence regarding your insurance policy to lorrainemurphy76@hotmail.com. Thanks, Chill Insurance.
Hi Ann 